# 01 · Depuración e integración de datos

**Pregunta del proyecto:** ¿en cuál de las tres zonas metropolitanas (Guadalajara, Monterrey o Querétaro) conviene abrir la próxima tienda Costco, y en qué municipio?

Este notebook construye las cuatro bases limpias que usan el modelo econométrico (02) y la evaluación financiera (03):

| Base de salida | Fuente oficial | Nivel |
|---|---|---|
| `hogares_enigh.csv` | INEGI, ENIGH 2022 (nueva serie): concentradohogar, hogares y gastoshogar | Hogar (90,102 hogares, todo el país) |
| `municipios_censo.csv` | INEGI, Censo de Población y Vivienda 2020 y 2010 (ITER) | Municipio (Jalisco, Nuevo León, Querétaro) |
| `clubes_precio.csv` | INEGI, DENUE (corte abril 2026) + comunicados y prensa para aperturas 2026 | Establecimiento |
| `zonas_metropolitanas.csv` | Delimitación de zonas metropolitanas (SEDATU, CONAPO, INEGI, *Metrópolis de México 2020*) | Municipio |

**Idea central:** la ENIGH registra en qué tipo de tienda compra cada hogar. El código `lugar_comp = 9` corresponde a **"Tiendas con membresía"** (Costco, Sam's Club, City Club). Con eso medimos directamente la demanda de clubes de precio. Después la llevamos a cada municipio usando *variables espejo*: características del hogar que aparecen igual en la ENIGH y en el Censo (automóvil, internet, computadora, horno de microondas, lavadora, TV de paga, tamaño del hogar y tamaño de localidad).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)
ESTADOS = {"14": "Jalisco", "19": "Nuevo León", "22": "Querétaro"}

## 1. ENIGH 2022: gasto de cada hogar en tiendas con membresía

`gasto_tri` es el gasto trimestral deflactado a agosto 2022. Lo multiplicamos por 4 para tenerlo anual.

In [2]:
cat = pd.read_csv(RAW / "enigh2022/cat_lugar_comp.csv", encoding="latin-1")
print(cat[cat.lugar_comp == 9])

g = pd.read_csv(RAW / "enigh2022/gastoshogar.csv", encoding="utf-8-sig",
                usecols=["folioviv", "foliohog", "lugar_comp", "gasto_tri"], dtype=str)
g["gasto_tri"] = pd.to_numeric(g["gasto_tri"], errors="coerce").fillna(0)
g["lugar_comp"] = pd.to_numeric(g["lugar_comp"], errors="coerce")
print(f"Registros de gasto: {len(g):,}")

por_lugar = g.groupby("lugar_comp")["gasto_tri"].sum().rename("gasto_tri")
club = (g[g.lugar_comp == 9].groupby(["folioviv", "foliohog"])["gasto_tri"].sum() * 4).rename("gasto_club")
super_ = (g[g.lugar_comp == 6].groupby(["folioviv", "foliohog"])["gasto_tri"].sum() * 4).rename("gasto_super")
print(f"Hogares con compras en tiendas con membresía: {len(club):,}")
del g

   lugar_comp            descripcion
9           9  Tiendas con membresía


Registros de gasto: 5,075,174


Hogares con compras en tiendas con membresía: 7,427


In [3]:
conc = pd.read_csv(RAW / "enigh2022/concentradohogar.csv", encoding="utf-8-sig", dtype={"folioviv": str, "foliohog": str, "ubica_geo": str},
                   usecols=["folioviv", "foliohog", "ubica_geo", "tam_loc", "est_socio", "upm", "est_dis", "factor",
                            "educa_jefe", "edad_jefe", "tot_integ", "ing_cor", "gasto_mon", "alimentos", "limpieza"])
hog = pd.read_csv(RAW / "enigh2022/hogares.csv", encoding="utf-8-sig", dtype={"folioviv": str, "foliohog": str},
                  usecols=["folioviv", "foliohog", "num_auto", "num_van", "num_pickup", "conex_inte", "tv_paga",
                           "num_compu", "num_micro", "num_lavad"])
df = (conc.merge(hog, on=["folioviv", "foliohog"], how="left", validate="1:1")
          .merge(club, on=["folioviv", "foliohog"], how="left")
          .merge(super_, on=["folioviv", "foliohog"], how="left"))
df[["gasto_club", "gasto_super"]] = df[["gasto_club", "gasto_super"]].fillna(0)

for c in ["num_auto", "num_van", "num_pickup", "num_compu", "num_micro", "num_lavad"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").clip(lower=0)

# Variables del modelo (todas también existen en el Censo 2020)
df["entidad"] = df["folioviv"].str[:2]
df["usa_club"] = (df["gasto_club"] > 0).astype(int)
df["auto"] = ((df.num_auto + df.num_van + df.num_pickup) > 0).astype(int)        # Censo: VPH_AUTOM
df["internet"] = (df["conex_inte"].astype(str) == "1").astype(int)              # Censo: VPH_INTER
df["computadora"] = (df["num_compu"] > 0).astype(int)                           # Censo: VPH_PC
df["microondas"] = (df["num_micro"] > 0).astype(int)                            # Censo: VPH_HMICRO
df["lavadora"] = (df["num_lavad"] > 0).astype(int)                              # Censo: VPH_LAVAD
df["tv_paga_d"] = (df["tv_paga"].astype(str) == "1").astype(int)                # Censo: VPH_STVP
df["tam_hogar"] = df["tot_integ"]                                               # Censo: PROM_OCUP
df["urbano_100k"] = (df["tam_loc"] == 1).astype(int)                            # Censo: % población en localidades de 100 mil o más
df["ing_anual"] = df["ing_cor"] * 4
df["gasto_anual"] = df["gasto_mon"] * 4
df["sin_dato"] = df[["auto", "internet"]].isna().any(axis=1)
print(f"Hogares: {len(df):,} | sin datos clave: {df.sin_dato.sum()}")

Hogares: 90,102 | sin datos clave: 0


### Control de calidad
Totales expandidos con el factor de expansión (`factor`). Todos los promedios y regresiones usan este ponderador.

In [4]:
w = df["factor"]
resumen = pd.DataFrame({
    "Hogares (millones)": [w.sum() / 1e6],
    "% hogares que compran en clubes": [np.average(df.usa_club, weights=w) * 100],
    "Gasto anual en clubes, total (miles de millones de pesos de 2022)": [(df.gasto_club * w).sum() / 1e9],
    "Gasto anual en supermercados, total (miles de millones)": [(df.gasto_super * w).sum() / 1e9],
}).T.round(2)
resumen.columns = ["Nacional"]
resumen

,Nacional
Hogares (millones),37.56
% hogares que compran en clubes,8.41
"Gasto anual en clubes, total (miles de millones de pesos de 2022)",67.99
"Gasto anual en supermercados, total (miles de millones)",588.80


In [5]:
def wmean(x, w):
    return np.average(x, weights=w)

por_estado = df.groupby("entidad").apply(lambda s: pd.Series({
    "hogares_muestra": len(s),
    "hogares_que_compran_muestra": s.usa_club.sum(),
    "pct_compra_club": wmean(s.usa_club, s.factor) * 100,
    "gasto_club_prom": wmean(s.gasto_club, s.factor),
    "ingreso_prom": wmean(s.ing_anual, s.factor),
    "pct_auto": wmean(s.auto, s.factor) * 100,
}), include_groups=False)
por_estado.loc[list(ESTADOS)].rename(index=ESTADOS).round(1)

,hogares_muestra,hogares_que_compran_muestra,pct_compra_club,gasto_club_prom,ingreso_prom,pct_auto
entidad,,,,,,
Jalisco,2625.0,198.0,8.6,1931.8,286977.7,54.2
Nuevo León,3612.0,240.0,9.6,1858.6,344072.1,57.7
Querétaro,3822.0,347.0,12.0,3244.0,299822.3,55.2


In [6]:
cols = ["folioviv", "foliohog", "entidad", "ubica_geo", "upm", "est_dis", "factor", "tam_loc", "est_socio",
        "usa_club", "gasto_club", "gasto_super", "ing_anual", "gasto_anual", "alimentos", "limpieza",
        "auto", "internet", "computadora", "microondas", "lavadora", "tv_paga_d", "tam_hogar", "urbano_100k",
        "educa_jefe", "edad_jefe"]
df[cols].to_csv(PROC / "hogares_enigh.csv", index=False)
print("Guardado hogares_enigh.csv", df[cols].shape)

Guardado hogares_enigh.csv (90102, 26)


## 2. Censo 2020 y 2010 por municipio

Del archivo ITER (Principales resultados por localidad) tomamos el renglón de total municipal (`LOC = 0000`). Las proporciones de equipamiento se calculan sobre viviendas particulares habitadas (`TVIVPARHAB`). La proporción urbana es la población que vive en localidades de 100 mil habitantes o más (equivalente a `tam_loc = 1` de la ENIGH).

In [7]:
VPH = {"VPH_AUTOM": "auto", "VPH_INTER": "internet", "VPH_PC": "computadora", "VPH_HMICRO": "microondas",
       "VPH_LAVAD": "lavadora", "VPH_STVP": "tv_paga_d"}
filas = []
for e in ESTADOS:
    it = pd.read_csv(RAW / f"censo2020/iter2020_{e}.csv", encoding="utf-8-sig", dtype=str)
    num = ["POBTOT", "TOTHOG", "TVIVPARHAB", "PROM_OCUP", "GRAPROES", "POCUPADA", "PDER_SS"] + list(VPH)
    for c in num:
        it[c] = pd.to_numeric(it[c], errors="coerce")
    loc = it[~it.LOC.isin(["0000", "9998", "9999"])].copy()
    urb = loc[loc.POBTOT >= 100_000].groupby("MUN").POBTOT.sum()
    mun = it[(it.LOC == "0000") & (it.MUN != "000")].copy()
    mun["pob_urbana_100k"] = mun.MUN.map(urb).fillna(0)
    filas.append(mun)
cen = pd.concat(filas)
cen["cve_mun"] = cen.ENTIDAD + cen.MUN
for k, v in VPH.items():
    cen[v] = cen[k] / cen["TVIVPARHAB"]
cen["tam_hogar"] = cen["PROM_OCUP"]
cen["urbano_100k"] = cen["pob_urbana_100k"] / cen["POBTOT"]
cen = cen.rename(columns={"NOM_MUN": "municipio", "POBTOT": "pob_2020", "TOTHOG": "hogares_2020", "GRAPROES": "escolaridad"})

c10 = pd.read_csv(RAW / "censo2010/iter2010_mun_14_19_22.csv", dtype=str)
c10["cve_mun"] = c10.ENTIDAD.str.zfill(2) + c10.MUN.str.zfill(3)
c10[["POBTOT", "TOTHOG"]] = c10[["POBTOT", "TOTHOG"]].apply(pd.to_numeric)
cen = cen.merge(c10[["cve_mun", "POBTOT", "TOTHOG"]].rename(columns={"POBTOT": "pob_2010", "TOTHOG": "hogares_2010"}),
                on="cve_mun", how="left")
# Tasa de crecimiento media anual de hogares 2010-2020 (INEGI); base para proyectar
cen["tcma_hogares"] = (cen.hogares_2020 / cen.hogares_2010) ** (1 / 10) - 1
cen["municipio"] = cen["municipio"].str.strip()
print(cen.shape, cen[list(VPH.values())].isna().sum().sum(), "celdas vacías")

(194, 299) 0 celdas vacías


## 3. Zonas metropolitanas (delimitación 2020)

Se identifican por nombre para no depender de claves capturadas a mano. Fuente: SEDATU, CONAPO e INEGI (2024), *Metrópolis de México 2020*. En Querétaro se excluye Apaseo el Alto (Guanajuato), que pertenece a la zona pero queda fuera de nuestra base estatal y pesa menos del 4% de su población.

In [8]:
ZM = {
    "Guadalajara": ("14", ["Guadalajara", "Zapopan", "San Pedro Tlaquepaque", "Tonalá", "Tlajomulco de Zúñiga", "El Salto",
                           "Juanacatlán", "Ixtlahuacán de los Membrillos", "Zapotlanejo", "Acatlán de Juárez"]),
    "Monterrey": ("19", ["Monterrey", "Guadalupe", "Apodaca", "San Nicolás de los Garza", "General Escobedo", "Juárez",
                         "García", "Santa Catarina", "San Pedro Garza García", "Cadereyta Jiménez", "Pesquería",
                         "Salinas Victoria", "Ciénega de Flores", "General Zuazua", "El Carmen", "Santiago", "Abasolo", "Hidalgo"]),
    "Querétaro": ("22", ["Querétaro", "Corregidora", "El Marqués", "Huimilpan"]),
}
zm_rows = []
for zm, (e, nombres) in ZM.items():
    sub = cen[cen.ENTIDAD == e]
    for n in nombres:
        hit = sub[sub.municipio == n]
        assert len(hit) == 1, f"No encontré {n} en {e}: {sub.municipio.tolist()[:5]}"
        zm_rows.append({"cve_mun": hit.cve_mun.iloc[0], "zona_metropolitana": zm})
zms = pd.DataFrame(zm_rows)
cen = cen.merge(zms, on="cve_mun", how="left")
cen.groupby("zona_metropolitana").agg(municipios=("cve_mun", "count"), poblacion_2020=("pob_2020", "sum"),
                                       hogares_2020=("hogares_2020", "sum"), hogares_2010=("hogares_2010", "sum"))

,municipios,poblacion_2020,hogares_2020,hogares_2010
zona_metropolitana,,,,
Guadalajara,10,5268642,1483691.0,1096489
Monterrey,18,5341177,1522975.0,1074122
Querétaro,4,1530820,447579.0,275842


In [9]:
cols_c = ["cve_mun", "ENTIDAD", "municipio", "zona_metropolitana", "pob_2010", "pob_2020", "hogares_2010", "hogares_2020",
          "tcma_hogares", "escolaridad", "POCUPADA", "PDER_SS"] + list(VPH.values()) + ["tam_hogar", "urbano_100k"]
cen[cols_c].rename(columns={"ENTIDAD": "entidad"}).to_csv(PROC / "municipios_censo.csv", index=False)
zms.to_csv(PROC / "zonas_metropolitanas.csv", index=False)
print("Guardado municipios_censo.csv y zonas_metropolitanas.csv")

Guardado municipios_censo.csv y zonas_metropolitanas.csv


## 4. Oferta: clubes de precio (DENUE + aperturas 2026)

En el DENUE los clubes están clasificados como SCIAN 462111 (supermercados). Los identificamos por nombre (Costco, Sam's, City Club) y exigimos más de 100 empleados para excluir oficinas, CEDIS y gasolineras. El corte del DENUE es abril de 2026, así que agregamos las aperturas recientes publicadas por la empresa y la prensa.

In [10]:
clubes = []
for e in ESTADOS:
    d = pd.read_csv(RAW / f"denue/denue_{e}.csv", encoding="latin-1", dtype=str,
                    usecols=["nom_estab", "raz_social", "codigo_act", "per_ocu", "cve_ent", "cve_mun", "municipio",
                             "latitud", "longitud", "fecha_alta"])
    txt = (d.nom_estab.fillna("") + " " + d.raz_social.fillna("")).str.upper()
    m = (d.codigo_act == "462111") & txt.str.contains(r"COSTCO|\bSAM'?S?\b|SAMS|CITY CLUB") & d.per_ocu.str.contains("101|251", na=False)
    k = d[m].copy()
    k["cadena"] = np.select([txt[m].str.contains("COSTCO"), txt[m].str.contains("CITY CLUB")], ["Costco", "City Club"], "Sam's Club")
    clubes.append(k)
cl = pd.concat(clubes)
cl["cve_mun"] = cl.cve_ent + cl.cve_mun
cl["fuente"] = "DENUE abr-2026"
cl["estatus"] = "abierto"

nuevas = pd.DataFrame([
    {"nom_estab": "COSTCO ESCOBEDO", "cadena": "Costco", "cve_mun": "19021", "municipio": "General Escobedo",
     "fuente": "Xataka / ADN40 (apertura 14-may-2026)", "estatus": "abierto"},
    {"nom_estab": "COSTCO CORREGIDORA", "cadena": "Costco", "cve_mun": "22006", "municipio": "Corregidora",
     "fuente": "ADN40 13-mar-2026; apertura en diciembre de 2026 (confirmado por el equipo)", "estatus": "abre dic-2026"},
])
cl = pd.concat([cl, nuevas], ignore_index=True)
cl = cl.merge(zms, on="cve_mun", how="left")
cl[["nom_estab", "cadena", "cve_mun", "municipio", "zona_metropolitana", "estatus", "fuente", "latitud", "longitud"]] \
    .to_csv(PROC / "clubes_precio.csv", index=False)
pd.crosstab(cl.zona_metropolitana.fillna("Fuera de las 3 zonas"), cl.cadena, margins=True)

cadena,City Club,Costco,Sam's Club,All
zona_metropolitana,,,,
Fuera de las 3 zonas,0,1,3,4
Guadalajara,1,3,9,13
Monterrey,5,4,10,19
Querétaro,1,2,3,6
All,7,10,25,42


**Lectura:** la zona de Monterrey tiene 19 clubes y la de Guadalajara 13. Querétaro tiene 6, contando la tienda de Corregidora anunciada. El modelo del notebook 02 dirá si esa diferencia en oferta corresponde a una diferencia en demanda.